## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-05-06-53-00 +0000,wsj,Brent Extends Declines on Hopes Mideast War Wi...,https://www.wsj.com/finance/commodities-future...
1,2026-11-05-06-49-00 +0000,wsj,ArcelorMittal Says Europe’s Higher Steel Tarif...,https://www.wsj.com/business/earnings/arcelorm...
2,2026-11-05-06-43-00 +0000,wsj,Commerzbank Flags Higher Deposit Competition a...,https://www.wsj.com/business/earnings/commerzb...
3,2026-11-05-06-35-25 +0000,bbc,"Joshua Wong, face of Hong Kong's pro-democracy...",https://www.bbc.co.uk/news/articles/cwp9g3rlp8...
4,2026-11-05-06-35-23 +0000,nypost,"California man, 97, dies at his own birthday p...",https://nypost.com/2026/11/05/us-news/californ...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2472/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
88,trump,61
54,midterm,41
72,democrats,39
278,midterms,32
55,election,24


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
219,2026-11-04-20-06-00 +0000,wsj,"Democrats won control of Congress, a majority ...",https://www.wsj.com/politics/elections/how-dem...,225
274,2026-11-04-18-06-00 +0000,wsj,"To win control of the House and Senate, Democr...",https://www.wsj.com/politics/elections/the-vot...,204
384,2026-11-04-11-34-05 +0000,chitri,Democrats win the US House and Senate as voter...,https://www.chicagotribune.com/2026/11/04/poll...,199
322,2026-11-04-15-51-00 +0000,wsj,The Democrats’ big victories in the House and ...,https://www.wsj.com/politics/elections/trump-p...,191
134,2026-11-04-22-37-00 +0000,wsj,Democrats’ midterm sweep has shaken up the 202...,https://www.wsj.com/politics/elections/democra...,182


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
219,225,2026-11-04-20-06-00 +0000,wsj,"Democrats won control of Congress, a majority ...",https://www.wsj.com/politics/elections/how-dem...
230,160,2026-11-04-19-49-18 +0000,nypost,"New Yorkers could be in charge of White House,...",https://nypost.com/2026/11/04/us-news/new-york...
134,102,2026-11-04-22-37-00 +0000,wsj,Democrats’ midterm sweep has shaken up the 202...,https://www.wsj.com/politics/elections/democra...
6,87,2026-11-05-06-13-59 +0000,nypost,Ted Cruz calls midterm election results a ‘Wat...,https://nypost.com/2026/11/05/us-news/ted-cruz...
138,83,2026-11-04-22-31-00 +0000,wsj,Vice President JD Vance blamed the war in Iran...,https://www.wsj.com/politics/policy/vances-bre...
239,68,2026-11-04-19-22-11 +0000,nypost,California voters overwhelmingly reject union-...,https://nypost.com/2026/11/04/us-news/californ...
243,56,2026-11-04-19-14-59 +0000,chitri,"Mayor Karen Bass, democratic socialist Nithya ...",https://www.chicagotribune.com/2026/11/04/kare...
44,50,2026-11-05-02-00-00 +0000,wsj,Americans’ votes for anti-data center candidat...,https://www.wsj.com/tech/ai/the-election-day-r...
414,45,2026-11-04-09-00-00 +0000,cbc,No catastrophic financial hit required to figh...,https://www.cbc.ca/news/science/climate-lawsui...
322,44,2026-11-04-15-51-00 +0000,wsj,The Democrats’ big victories in the House and ...,https://www.wsj.com/politics/elections/trump-p...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
